# Candidate Retrieval Demo

Given a SMILES string, retrieve:
1. **Formula-constrained candidates** from PubChem (all same molecular formula, ranked by Tanimoto, top-50 non-stereo SMILES)
2. **RI-based candidates** from PubChem (closest predicted retention index, arbitrary N)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import h5py
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from icicle.utils import get_morgan_fp_from_smi

RDLogger.DisableLog("rdApp.*")

# --- Config ---
SMILES = "O=C(CBr)NC1CCCCC1"
TOP_N_FORMULA = (
    50  # max non-stereo candidates from formula match (matches eval pipeline)
)
TOP_N_RI = 50  # number of RI-closest candidates

FORMULA_MAP_PATH = "../../../data/PubChem/pubchem_formula_map.p"
# HDF5 contains AIRI-predicted RI (ri_StdNP/SemiStdNP/StdPolar), smiles,
# inchikey14, sort_idx, sorted_ri — all precomputed from batch inference
PUBCHEM_HDF5_PATH = (
    "../../../data/PubChem/pubchem_predictions_with_ik14_master.hdf5"
)
IK14_INDEX_PATH = (
    "../../../data/PubChem/pubchem_predictions_with_ik14_master_ik14_index.npy"
)

In [ ]:
# Compute molecule properties
mol = Chem.MolFromSmiles(SMILES)
inchi = Chem.MolToInchi(mol)
inchikey = Chem.InchiToInchiKey(inchi)
inchikey14 = inchikey[:14]
formula = rdMolDescriptors.CalcMolFormula(mol)
ns_smiles = Chem.MolToSmiles(Chem.MolFromSmiles(SMILES), isomericSmiles=False)

print(f"SMILES:       {SMILES}")
print(f"InChIKey:     {inchikey}")
print(f"InChIKey-14:  {inchikey14}")
print(f"Formula:      {formula}")
print(f"Non-stereo:   {ns_smiles}")

## 1. Formula-Constrained Candidates

All PubChem molecules with the same molecular formula, ranked by Tanimoto similarity (Morgan FP), capped at `TOP_N_FORMULA` non-stereo SMILES. Stereo variants of each kept. This matches the eval pipeline in `retrieval_create_retrieval_lists.py`.

In [ ]:
print("Loading PubChem formula map (this may take ~30s)...")
with open(FORMULA_MAP_PATH, "rb") as f:
    formula_map = pickle.load(f)
print("Done.")

In [ ]:
entry = formula_map.get(formula, {})
print(
    f"Raw formula matches for {formula}: {sum(len(v) for v in entry.values())} stereo isomers "
    f"across {len(entry)} non-stereo scaffolds"
)

# Rank non-stereo scaffolds by Tanimoto, keep top-N (matches eval pipeline)
query_fp = get_morgan_fp_from_smi(ns_smiles)
ranked = []
for cand_ns_smi, stereo_set in entry.items():
    if not stereo_set:
        continue
    cand_fp = get_morgan_fp_from_smi(cand_ns_smi)
    intersect = np.dot(query_fp, cand_fp)
    union = query_fp.sum() + cand_fp.sum() - intersect
    tani = float(intersect / (union + 1e-22))
    for stereo_smi, ik in stereo_set:
        ranked.append(
            {
                "tanimoto": tani,
                "ns_smiles": cand_ns_smi,
                "smiles": stereo_smi,
                "inchikey": ik,
            }
        )

# Sort by Tanimoto descending, then cap at top-N *non-stereo* scaffolds
df_all = pd.DataFrame(ranked).sort_values("tanimoto", ascending=False)
top_ns = df_all["ns_smiles"].unique()[:TOP_N_FORMULA]
df_formula = df_all[df_all["ns_smiles"].isin(top_ns)].reset_index(drop=True)

print(
    f"\nTop-{TOP_N_FORMULA} non-stereo scaffolds → {len(df_formula)} candidates (incl. stereo variants)"
)
print(f"Query molecule in set: {inchikey in df_formula['inchikey'].values}")
df_formula[["tanimoto", "smiles", "inchikey"]].head(20)

## 2. RI-Based Candidates

Predict RI for the query molecule, then retrieve the `TOP_N_RI` PubChem molecules with closest predicted StdNP retention index. Uses binary search on the pre-sorted RI arrays in the HDF5 — fast, no full scan.

In [ ]:
# AIRI RI predictions are already stored in the HDF5 from batch inference.
# Load the ik14 index (dict: inchikey14 str -> row index) for O(1) lookup.
ik14_index = np.load(IK14_INDEX_PATH, allow_pickle=True).item()

row = ik14_index.get(inchikey14)
if row is None:
    raise KeyError(
        f"{inchikey14} not found in PubChem HDF5 — molecule may not be in PubChem"
    )

with h5py.File(PUBCHEM_HDF5_PATH, "r") as f:
    query_ri = float(f["ri_StdNP"][row])

print(f"Predicted RI (StdNP, AIRI): {query_ri:.2f}")

In [ ]:
# HDF5 has pre-sorted RI arrays + sort_idx → binary search, no full scan needed
with h5py.File(PUBCHEM_HDF5_PATH, "r") as f:
    sorted_ri = f["sorted_ri_StdNP"][:]  # sorted RI values
    sort_idx = f["sort_idx_StdNP"][:]  # original row indices in sorted order

idx_center = np.searchsorted(sorted_ri, query_ri)
half = TOP_N_RI // 2
lo = max(0, idx_center - half)
hi = min(len(sorted_ri), idx_center + half)

# Expand window until we have exactly TOP_N_RI rows
while (hi - lo) < TOP_N_RI and (lo > 0 or hi < len(sorted_ri)):
    if lo > 0:
        lo -= 1
    if hi < len(sorted_ri) and (hi - lo) < TOP_N_RI:
        hi += 1

orig_indices = sort_idx[lo:hi]
ri_window = sorted_ri[lo:hi]

# Fetch smiles + inchikey14 for those rows from HDF5
with h5py.File(PUBCHEM_HDF5_PATH, "r") as f:
    smiles_all = f["smiles"]
    inchikey14_all = f["inchikey14"]
    smiles_vals = np.array([smiles_all[i].decode() for i in orig_indices])
    ik14_vals = np.array([inchikey14_all[i].decode() for i in orig_indices])

df_ri = (
    pd.DataFrame(
        {
            "smiles": smiles_vals,
            "inchikey14": ik14_vals,
            "ri_StdNP": ri_window,
            "ri_diff": np.abs(ri_window - query_ri),
        }
    )
    .sort_values("ri_diff")
    .head(TOP_N_RI)
    .reset_index(drop=True)
)
df_ri.index += 1
df_ri.index.name = "rank"

print(f"Top-{TOP_N_RI} RI-based candidates (query_ri={query_ri:.1f}, StdNP):")
print(
    f"RI range: {df_ri['ri_StdNP'].min():.2f} – {df_ri['ri_StdNP'].max():.2f}"
)
print(f"Max RI diff: {df_ri['ri_diff'].max():.4f}")
df_ri